In [5]:
from torch import optim
from torchvision.models import efficientnet_b7, EfficientNet_B7_Weights
import torch
import random
import numpy as np
import torch.nn as nn
import albumentations as Albu
import pandas as pd
from torch.utils.data.sampler import RandomSampler
from warmup_scheduler import GradualWarmupScheduler
import os
from utils.dataset import PandasDataset
from utils.metrics import model_checkpoint
from utils.train import train_model
from utils.models import EfficientNetApi

In [6]:
seed = 42
shuffle = True
batch_size = 2
num_workers = 4
output_classes = 5
init_lr = 3e-4
warmup_factor = 2
warmup_epochs = 1
n_epochs = 50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
loss_function = nn.BCEWithLogitsLoss()

torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

ROOT_DIR = '../../..'

data_dir = '../../../../dataset'
images_dir = os.path.join(data_dir, 'tiles')

Using device: cuda


In [7]:
load_model = efficientnet_b7(
     weights=EfficientNet_B7_Weights.DEFAULT
)
model = EfficientNetApi(model=load_model, output_dimensions=output_classes, dropout_rate=0.6)
model = model.to(device)

In [8]:
print("Using device:", device)
loss_function = nn.BCEWithLogitsLoss()

torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

Using device: cuda


In [9]:
df_train_ = pd.read_csv(f"{ROOT_DIR}/data/train_5fold.csv")
df_train_.columns = df_train_.columns.str.strip()
train_indexes = np.where((df_train_['fold'] != 3))[0]
valid_indexes = np.where((df_train_['fold'] == 3))[0]
#
df_train = df_train_.loc[train_indexes]
df_val = df_train_.loc[valid_indexes]
df_test = pd.read_csv(f"{ROOT_DIR}/data/test.csv")

#### view data

In [10]:
(df_train.shape, df_val.shape, df_test.shape)

((7219, 5), (1805, 5), (1592, 4))

In [11]:
transforms = Albu.Compose([
    Albu.Transpose(p=0.5),
    Albu.VerticalFlip(p=0.5),
    Albu.HorizontalFlip(p=0.5),
])

In [12]:
df_train.columns = df_train.columns.str.strip()

train_dataset = PandasDataset(images_dir, df_train, transforms=transforms)
valid_dataset = PandasDataset(images_dir, df_val, transforms=None)
test_dataset = PandasDataset(images_dir, df_test, transforms=None)

In [13]:
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, num_workers=num_workers, sampler=RandomSampler(train_dataset)
)
valid_loader = torch.utils.data.DataLoader(
    valid_dataset, batch_size=batch_size, num_workers=num_workers, sampler = RandomSampler(valid_dataset)
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=batch_size, num_workers=num_workers, sampler = RandomSampler(test_dataset)
)

In [14]:
optimizer = optim.Adam(model.parameters(), lr = init_lr / warmup_factor)
scheduler_cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, n_epochs - warmup_epochs)
scheduler = GradualWarmupScheduler(optimizer, multiplier = warmup_factor, total_epoch = warmup_epochs, after_scheduler=scheduler_cosine)

In [11]:
train_model(
    model=model,
    epochs=n_epochs,
    optimizer=optimizer,
    scheduler=scheduler,
    train_dataloader=train_loader,
    valid_dataloader=valid_loader,
    checkpoint=model_checkpoint,
    device=device,
    loss_function=loss_function,
    path_to_save_metrics="logs/b7.txt",
    path_to_save_model="models/b7.pth",
    patience=5,
)

Epoch 1/50



100%|██████████| 903/903 [07:31<00:00,  2.00it/s]


VAL_LOSS     0.331
VAL_ACC      Mean: 38.815 | Std: 1.180 | 95% CI: [36.898, 40.720]
VAL_KAPPA    Mean: 0.726 | Std: 0.012 | 95% CI: [0.705, 0.746]
VAL_F1       Mean: 0.385 | Std: 0.012 | 95% CI: [0.365, 0.405]
VAL_RECALL   Mean: 0.389 | Std: 0.012 | 95% CI: [0.369, 0.408]
VAL_PRECISION Mean: 0.452 | Std: 0.013 | 95% CI: [0.431, 0.473]
Salvando o melhor modelo... 0.0 -> 0.7258552745561231
Epoch 2/50



100%|██████████| 903/903 [07:32<00:00,  2.00it/s]
/home/woshington/Projects/Doutorado/repo/.venv/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:1087: UserWarning: To get the last learning rate computed by the scheduler, please use `get_last_lr()`.
  _warn_get_lr_called_within_step(self)


VAL_LOSS     0.350
VAL_ACC      Mean: 45.135 | Std: 1.211 | 95% CI: [43.102, 47.036]
VAL_KAPPA    Mean: 0.701 | Std: 0.014 | 95% CI: [0.677, 0.722]
VAL_F1       Mean: 0.378 | Std: 0.012 | 95% CI: [0.359, 0.398]
VAL_RECALL   Mean: 0.388 | Std: 0.011 | 95% CI: [0.369, 0.406]
VAL_PRECISION Mean: 0.486 | Std: 0.019 | 95% CI: [0.456, 0.516]
Epoch 3/50



...

VAL_LOSS     0.713
VAL_ACC      Mean: 55.723 | Std: 1.213 | 95% CI: [53.684, 57.621]
VAL_KAPPA    Mean: 0.763 | Std: 0.014 | 95% CI: [0.740, 0.786]
VAL_F1       Mean: 0.487 | Std: 0.013 | 95% CI: [0.466, 0.508]
VAL_RECALL   Mean: 0.477 | Std: 0.012 | 95% CI: [0.459, 0.497]
VAL_PRECISION Mean: 0.518 | Std: 0.014 | 95% CI: [0.495, 0.541]
Epoch 15/50



100%|██████████| 903/903 [07:29<00:00,  2.01it/s]


VAL_LOSS     0.681
VAL_ACC      Mean: 54.209 | Std: 1.177 | 95% CI: [52.296, 56.180]
VAL_KAPPA    Mean: 0.793 | Std: 0.013 | 95% CI: [0.773, 0.814]
VAL_F1       Mean: 0.496 | Std: 0.012 | 95% CI: [0.476, 0.516]
VAL_RECALL   Mean: 0.496 | Std: 0.012 | 95% CI: [0.476, 0.516]
VAL_PRECISION Mean: 0.507 | Std: 0.013 | 95% CI: [0.485, 0.527]
Epoch 16/50



100%|██████████| 903/903 [07:29<00:00,  2.01it/s]


VAL_LOSS     0.663
VAL_ACC      Mean: 55.889 | Std: 1.159 | 95% CI: [54.017, 57.839]
VAL_KAPPA    Mean: 0.787 | Std: 0.013 | 95% CI: [0.767, 0.809]
VAL_F1       Mean: 0.505 | Std: 0.013 | 95% CI: [0.484, 0.527]
VAL_RECALL   Mean: 0.503 | Std: 0.012 | 95% CI: [0.482, 0.523]
VAL_PRECISION Mean: 0.523 | Std: 0.013 | 95% CI: [0.501, 0.546]

Early stopping at epoch 16. No improvement for 5 epochs.
Best epoch: 11 with kappa: 0.8034


# tests

In [15]:
from utils.metrics import evaluation, format_metrics
model.load_state_dict(
    torch.load(f"models/b7.pth")
)
response = evaluation(model, test_loader, device)
result = format_metrics(response[0])
print(result)

100%|██████████| 796/796 [06:33<00:00,  2.02it/s]


VAL_ACC      Mean: 56.753 | Std: 1.257 | 95% CI: [54.585, 58.794]
VAL_KAPPA    Mean: 0.804 | Std: 0.013 | 95% CI: [0.783, 0.825]
VAL_F1       Mean: 0.499 | Std: 0.013 | 95% CI: [0.477, 0.520]
VAL_RECALL   Mean: 0.499 | Std: 0.013 | 95% CI: [0.478, 0.520]
VAL_PRECISION Mean: 0.504 | Std: 0.013 | 95% CI: [0.481, 0.526]
